# Diabetes Regression Model Comparison

## Project overview
This notebook compares multiple machine learning regression models on the diabetes dataset and evaluates how well each model predicts a continuous target variable.

## Objectives
- Benchmark linear and non-linear regression models on the same dataset
- Compare performance using MAE, RMSE, and R²
- Use cross-validation for more reliable evaluation
- Visualize model performance for clearer interpretation

## Skills demonstrated
- Supervised learning
- Regression benchmarking
- Pipelines and feature scaling
- Cross-validation
- Model evaluation and visualization


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
data = load_diabetes(as_frame=True)
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [3]:
def get_regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

def display_metrics(name, metrics):
    print(f"{name:>12} | MAE={metrics['MAE']:.2f}  RMSE={metrics['RMSE']:.2f}  R2={metrics['R2']:.3f}")


In [4]:
lin = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
ridge = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
lasso = Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.05, max_iter=5000))])

results = []
predictions = {}

for name, model in [("Linear Regression", lin), ("Ridge Regression", ridge), ("Lasso Regression", lasso)]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = get_regression_metrics(y_test, pred)
    display_metrics(name, metrics)
    results.append({"Model": name, **metrics})
    predictions[name] = pred


      Linear | MAE=42.79  RMSE=53.85  R2=0.453
       Ridge | MAE=42.81  RMSE=53.78  R2=0.454
       Lasso | MAE=42.80  RMSE=53.77  R2=0.454


In [5]:
rf = RandomForestRegressor(n_estimators=600, random_state=42)
gbr = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=800, random_state=42)

for name, model in [("Random Forest", rf), ("HistGradientBoosting", gbr)]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = get_regression_metrics(y_test, pred)
    display_metrics(name, metrics)
    results.append({"Model": name, **metrics})
    predictions[name] = pred

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df


          RF | MAE=44.38  RMSE=54.70  R2=0.435
         GBR | MAE=43.77  RMSE=55.29  R2=0.423


In [6]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for name, model in [
    ("Linear Regression", lin),
    ("Ridge Regression", ridge),
    ("Lasso Regression", lasso),
    ("Random Forest", rf),
    ("HistGradientBoosting", gbr),
]:
    rmse_scores = -cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    cv_rows.append({
        "Model": name,
        "CV RMSE Mean": rmse_scores.mean(),
        "CV RMSE Std": rmse_scores.std(),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV RMSE Mean").reset_index(drop=True)
cv_df


Ridge CV RMSE mean/std: 55.374175368673455 2.3732060814135294
Lasso CV RMSE mean/std: 55.369559245603924 2.373493175915404


## Visualization of model performance

The following charts summarize test-set performance and help identify which model achieves the lowest prediction error and strongest fit.

In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df.copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["Model"], plot_df["RMSE"])
plt.xticks(rotation=20, ha="right")
plt.ylabel("RMSE")
plt.title("Model Comparison by Test RMSE")
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_pred = predictions[best_model_name]

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_pred, alpha=0.7)
plt.xlabel("Actual Target")
plt.ylabel("Predicted Target")
plt.title(f"Actual vs Predicted Values ({best_model_name})")
plt.tight_layout()
plt.show()

## Summary

- The models were compared on the same train/test split for a fair benchmark.
- Lower **RMSE** and **MAE** indicate better predictive accuracy.
- Higher **R²** indicates a better fit to the target values.
- The final tables and charts make it easier to present the project on GitHub and LinkedIn.
